In [ ]:
"""Simulation 1 — complete-data CTMC MLE, reviewer-focused diagnostics (CPU) v4.

Main experiments retain complete-data MLE and exact CTMC final-size risk.

Improvements over v3: a predeclared decision-threshold subset, separately calibrated
simultaneous bands on this subset, Gaussian max statistic calibration,
threshold-wise variance diagnostics, and logit-scale subset sensitivity.

Retained features:
- Original bootstrap max-t validates resamples using original-data SE only.
- Re-studentized bootstrap has its own validity filter and cannot invalidate other methods.
- Positive log-rate linearization diagnostic (NOT the original bootstrap method).
- Empirical versus model-estimated parameter covariance report across Monte Carlo replicates.
- Reports zero-count/boundary resamples, extreme T-values, unconditional failures.
- N=50, m=(50,100,250,500), B=999; use MC=2000 for precise validation.
- Exact full-path CTMC simulation, closed-form complete-data MLE,
  exact final-size recursions and parametric re-simulation/refitting.

Paste this entire code into one Colab cell, or upload companion one-cell .ipynb.
"""
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
import pathlib, sys, math, time, pickle, json, argparse, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
import numba
from numba import njit, prange
from scipy.stats import norm
from scipy.special import expit

BASE_THETA=np.array([1.45,.75,1.,1.,.35,.25],dtype=np.float64)
DEFAULT_M=[50,100,250,500]
PARAM_NAMES=['beta1','beta2','gamma1','gamma2','q12','q21']
METHODS=['Pointwise Wald','Bonferroni','Gaussian max-t',
         'Bootstrap max-t','Bootstrap max-t (restudentized)',
         'Log-rate linearized bootstrap max-t']
SUBSET_LOGIT_METHOD='Logit-scale bootstrap (subset)'
# Fixed ex ante: proportions of the *initial susceptible* population.
# These are scientific threshold comparisons, not a data-adaptive selection.
THRESHOLD_FRACTIONS=(0.10,0.20,0.40,0.60)
LOGIT_EPS=1e-7

@njit(cache=False)
def _one_dataset(m, theta, N, i0, seed):
    """Exact full continuous-time trajectories, summarized by six sufficient statistics."""
    np.random.seed(seed)
    cnt = np.zeros(6, dtype=np.float64)
    expo = np.zeros(6, dtype=np.float64)
    q12, q21 = theta[4], theta[5]
    pi0 = q21/(q12+q21)
    for j in range(m):
        s, i = N-i0, i0
        p = 0 if np.random.random() < pi0 else 1
        while i > 0:
            base = s*i/N
            inf = theta[p]*base
            rec = theta[p+2]*i
            sw = theta[p+4]
            rate = inf+rec+sw
            wait = -math.log(max(np.random.random(), 1e-300))/rate
            expo[p] += base*wait
            expo[2+p] += i*wait
            expo[4+p] += wait
            u = np.random.random()*rate
            if u < inf:
                s -= 1
                i += 1
                cnt[p] += 1.0
            elif u < inf+rec:
                i -= 1
                cnt[2+p] += 1.0
            else:
                p = 1-p
                cnt[4+(1-p)] += 1.0 # from old p
    return cnt, expo

@njit(cache=False, parallel=True)
def _boot_mles(m, B, theta, N, i0, seeds):
    fits = np.empty((B, 6), dtype=np.float64)
    for k in prange(B):
        c, e = _one_dataset(m, theta, N, i0, seeds[k])
        for j in range(6):
            fits[k, j] = c[j]/e[j] if e[j] > 0 else np.nan
    return fits

@njit(cache=False, parallel=True)
def _boot_mles_counts(m, B, theta, N, i0, seeds):
    """Parametric bootstrap: full re-simulation and exact complete-data refit.

    Preserve event counts for the bootstrap-specific information matrix.
    """
    fits = np.empty((B, 6), dtype=np.float64)
    counts = np.empty((B, 6), dtype=np.float64)
    for k in prange(B):
        c, e = _one_dataset(m, theta, N, i0, seeds[k])
        for j in range(6):
            counts[k, j] = c[j]
            fits[k, j] = c[j]/e[j] if e[j] > 0 else np.nan
    return fits, counts

@njit(cache=False)
def _risk_curve(theta, N, i0):
    """Exact final-size tail curve at specified regime 1 (p=0)."""
    smax = N-i0
    b1, b2, g1, g2, q12, q21 = theta
    x = np.zeros((smax+1, N+1, smax+1, 2), dtype=np.float64)
    for s in range(smax+1):
        for i in range(N-s+1):
            if i == 0 or s == 0:
                x[s,i,0,0] = 1.0
                x[s,i,0,1] = 1.0
                continue
            t = s*i/N
            r1, r2 = b1*t, b2*t
            rec1, rec2 = g1*i, g2*i
            a, d = r1+rec1+q12, r2+rec2+q21
            det = a*d-q12*q21
            for n in range(s+1):
                h1 = rec1*x[s,i-1,n,0]
                h2 = rec2*x[s,i-1,n,1]
                if n > 0:
                    h1 += r1*x[s-1,i+1,n-1,0]
                    h2 += r2*x[s-1,i+1,n-1,1]
                x[s,i,n,0] = (d*h1+q12*h2)/det
                x[s,i,n,1] = (q21*h1+a*h2)/det
    risk = np.empty(smax, dtype=np.float64)
    tail = 0.0
    for n in range(smax,0,-1):
        tail += x[smax,i0,n,0]
        risk[n-1] = tail
    return risk

@njit(cache=False)
def _risk_and_gradient(theta, N, i0):
    """Exact final-size recursion with all six analytic sensitivities."""
    smax = N-i0
    b1, b2, g1, g2, q12, q21 = theta
    x = np.zeros((smax+1, N+1, smax+1, 2), dtype=np.float64)
    grad = np.zeros((smax+1, N+1, smax+1, 2, 6), dtype=np.float64)
    for s in range(smax+1):
        for i in range(N-s+1):
            if i == 0 or s == 0:
                x[s,i,0,0] = 1.0
                x[s,i,0,1] = 1.0
                continue
            t = s*i/N
            r1, r2 = b1*t, b2*t
            rec1, rec2 = g1*i, g2*i
            a, d = r1+rec1+q12, r2+rec2+q21
            det = a*d-q12*q21
            for n in range(s+1):
                h1 = rec1*x[s,i-1,n,0]
                h2 = rec2*x[s,i-1,n,1]
                if n > 0:
                    h1 += r1*x[s-1,i+1,n-1,0]
                    h2 += r2*x[s-1,i+1,n-1,1]
                u1 = (d*h1+q12*h2)/det
                u2 = (q21*h1+a*h2)/det
                x[s,i,n,0] = u1
                x[s,i,n,1] = u2
                for j in range(6):
                    dh1 = rec1*grad[s,i-1,n,0,j]
                    dh2 = rec2*grad[s,i-1,n,1,j]
                    if n > 0:
                        dh1 += r1*grad[s-1,i+1,n-1,0,j]
                        dh2 += r2*grad[s-1,i+1,n-1,1,j]
                        if j == 0:
                            dh1 += t*x[s-1,i+1,n-1,0]
                        elif j == 1:
                            dh2 += t*x[s-1,i+1,n-1,1]
                    if j == 0:
                        dh1 -= t*u1
                    elif j == 1:
                        dh2 -= t*u2
                    elif j == 2:
                        dh1 += i*(x[s,i-1,n,0]-u1)
                    elif j == 3:
                        dh2 += i*(x[s,i-1,n,1]-u2)
                    elif j == 4:
                        dh1 -= u1-u2
                    else:
                        dh2 -= u2-u1
                    grad[s,i,n,0,j] = (d*dh1+q12*dh2)/det
                    grad[s,i,n,1,j] = (q21*dh1+a*dh2)/det
    risk = np.empty(smax, dtype=np.float64)
    jac = np.empty((smax,6), dtype=np.float64)
    tail = 0.0
    ders = np.zeros(6, dtype=np.float64)
    for n in range(smax,0,-1):
        tail += x[smax,i0,n,0]
        for j in range(6):
            ders[j] += grad[smax,i0,n,0,j]
            jac[n-1,j] = ders[j]
        risk[n-1] = tail
    return risk, jac

@njit(cache=False, parallel=True)
def _boot_risks(fits, N, i0):
    B = fits.shape[0]
    smax = N-i0
    curves = np.empty((B,smax), dtype=np.float64)
    for k in prange(B):
        curves[k,:] = _risk_curve(fits[k,:], N, i0)
    return curves


@njit(cache=False, parallel=True)
def _boot_risk_ses(fits, counts, N, i0):
    """Exact bootstrap risk and per-bootstrap delta-method SE at ALL fixed thresholds.

    Boundary fits are not silently repaired or counted as valid: NaN marks the
    undefined bootstrap studentization. Runs across Numba CPU workers.
    """
    B = fits.shape[0]
    s = N-i0
    curves = np.empty((B, s), dtype=np.float64)
    ses = np.empty((B, s), dtype=np.float64)
    for k in prange(B):
        fits_ok = True
        for j in range(6):
            if not (np.isfinite(fits[k,j]) and fits[k,j] > 0 and counts[k,j] > 0):
                fits_ok = False
        if fits_ok:
            curve, jac = _risk_and_gradient(fits[k], N, i0)
            for h in range(s):
                curves[k,h] = curve[h]
                variance = 0.0
                for j in range(6):
                    variance += jac[h,j]*jac[h,j]*fits[k,j]*fits[k,j]/counts[k,j]
                ses[k,h] = math.sqrt(max(variance, 0.0))
        else:
            for h in range(s):
                curves[k,h] = np.nan
                ses[k,h] = np.nan
    return curves, ses


@njit(cache=False, parallel=True)
def _boot_risks_allow_boundary(fits, N, i0):
    """Evaluate all *finite nonnegative* boundary MLEs, if CTMC recursion remains nonsingular.

    Original fixed-SE max-t does not require any bootstrap-specific SE. A fit with
    an observed count of zero is a legitimate boundary MLE, not automatically invalid.
    """
    B=fits.shape[0]
    s=N-i0
    result=np.full((B,s),np.nan)
    for k in prange(B):
        good=True
        for j in range(6):
            if not (np.isfinite(fits[k,j]) and fits[k,j]>=0):
                good=False
        if good:
            # With positive recovery and positive switching, this always holds.
            # Avoid singular 2x2 blocks if bootstrap has absorbing phase.
            beta1,beta2,g1,g2,q12,q21=fits[k]
            det0=(beta1+g1+q12)*(beta2+g2+q21)-q12*q21
            if det0>1e-14:
                curve=_risk_curve(fits[k],N,i0)
                ok=True
                for h in range(s):
                    if not np.isfinite(curve[h]):
                        ok=False
                if ok:
                    for h in range(s):
                        result[k,h]=curve[h]
    return result


def _finite_q(values,alpha):
    """Conservative, finite-sample order statistic at ceil((n+1)*(1-alpha))."""
    values=np.asarray(values,dtype=float)
    values=values[np.isfinite(values)]
    n=len(values)
    if n==0: return np.nan
    rank=min(n,math.ceil((n+1)*(1-alpha)))
    return float(np.partition(values,rank-1)[rank-1])


def _gaussian_critical(J,cov,se,gaussian_draws,alpha):
    # The estimated information matrix is diagonal in the six rate coordinates.
    delta=gaussian_draws*np.sqrt(np.diag(cov))[None,:]
    t=np.max(np.abs((delta@J.T)/se[None,:]),axis=1)
    return _finite_q(t,alpha)


def _atomic_save(path,obj):
    tmp=path.with_suffix('.tmp')
    with tmp.open('wb') as f:
        pickle.dump(obj,f,protocol=pickle.HIGHEST_PROTOCOL)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)


def _save(state,rng,path,root):
    state['rng_state']=rng.bit_generator.state
    _atomic_save(path,state)
    # Writing CSV on every checkpoint is avoided; Drive I/O is expensive.


def _start_or_resume(root,cfg,mode,seed):
    root.mkdir(parents=True,exist_ok=True)
    path=root/'checkpoint.pkl'; config_file=root/'config.json'
    serial=json.dumps(cfg,sort_keys=True)
    if mode=='new':
        if path.exists():
            raise FileExistsError('Checkpoint already exists; choose RESUME or a new folder.')
        rng=np.random.default_rng(seed)
        state={'rows':[],'rng_state':rng.bit_generator.state}
        config_file.write_text(serial)
        _atomic_save(path,state)
    else:
        if not path.exists() or not config_file.exists():
            raise FileNotFoundError('Checkpoint not found in '+str(root))
        if config_file.read_text()!=serial:
            raise ValueError('Checkpoint settings differ from requested options')
        with path.open('rb') as f:state=pickle.load(f)
        rng=np.random.default_rng()
        rng.bit_generator.state=state['rng_state']
    return state,rng,path


def _subset_indices(N,i0,fractions=THRESHOLD_FRACTIONS):
    s=N-i0
    c=np.unique(np.clip(np.rint(s*np.asarray(fractions)).astype(int),1,s))
    if len(c)<2:
        raise ValueError('Need at least two distinct prespecified thresholds')
    return c-1,c


def _metrics_for_critical(row,method,q,hat,truth,se,tail):
    ok=bool(np.isfinite(q) and q>0)
    row['valid_'+method]=ok
    row['q_'+method]=float(q) if ok else np.nan
    if not ok:
        for metric in ('full','tail','width','clipped_width','point','missed'):
            row[method+'_'+metric]=np.nan
        return
    err=np.abs(hat-truth)
    cover=(err<=q*se)
    lower=np.maximum(0.,hat-q*se)
    upper=np.minimum(1.,hat+q*se)
    row[method+'_full']=int(np.all(cover))
    row[method+'_tail']=int(np.all(cover[tail]))
    row[method+'_point']=float(np.mean(cover))
    row[method+'_missed']=int(np.sum(~cover))
    row[method+'_width']=float(2*q*np.mean(se))
    row[method+'_clipped_width']=float(np.mean(upper-lower))


def _subset_band(row,method,q,hat,truth,se):
    good=bool(np.isfinite(q) and q>0)
    row['valid_subset_'+method]=good
    row['q_subset_'+method]=float(q) if good else np.nan
    if not good:
        for key in ('full','width','clipped_width','point','missed'):
            row[method+'_subset_'+key]=np.nan
        return
    err=np.abs(hat-truth)
    inside=err<=q*se
    lo=np.maximum(0.,hat-q*se)
    hi=np.minimum(1.,hat+q*se)
    row[method+'_subset_full']=int(inside.all())
    row[method+'_subset_width']=float(2*q*np.mean(se))
    row[method+'_subset_clipped_width']=float(np.mean(hi-lo))
    row[method+'_subset_point']=float(inside.mean())
    row[method+'_subset_missed']=int((~inside).sum())


def _logit(x):
    p=np.clip(np.asarray(x,dtype=float),LOGIT_EPS,1.-LOGIT_EPS)
    return np.log(p)-np.log1p(-p)


def _logit_subset_band(row,q,hat,truth,se):
    method=SUBSET_LOGIT_METHOD
    ok=bool(np.isfinite(q) and q>0)
    row['valid_subset_'+method]=ok
    row['q_subset_'+method]=float(q) if ok else np.nan
    if not ok:
        for key in ('full','width','clipped_width','point','missed'):
            row[method+'_subset_'+key]=np.nan
        return
    p=np.clip(hat,LOGIT_EPS,1.-LOGIT_EPS)
    z=_logit(p)
    sez=se/(p*(1-p))
    lo=expit(z-q*sez)
    hi=expit(z+q*sez)
    inside=(truth>=lo)&(truth<=hi)
    row[method+'_subset_full']=int(inside.all())
    row[method+'_subset_width']=float(np.mean(hi-lo))
    row[method+'_subset_clipped_width']=float(np.mean(hi-lo))
    row[method+'_subset_point']=float(inside.mean())
    row[method+'_subset_missed']=int((~inside).sum())


def _one_replication(theta0,risk0,m,N,i0,B,alpha,rng,gaussian_draws,
                     min_boot_frac=0.90):
    t0=time.perf_counter()
    row=dict(valid=False,reason='',m=m)
    counts,exposures=_one_dataset(m,theta0,N,i0,int(rng.integers(1,2**31-1)))
    row['minimum_original_count']=int(np.min(counts))
    for j,name in enumerate(PARAM_NAMES):
        row['count_'+name]=int(counts[j]);row['exposure_'+name]=float(exposures[j])
    if np.any(counts<=0) or np.any(exposures<=0):
        row['reason']='original MLE on boundary / no exposure'
        row['wall_seconds']=time.perf_counter()-t0
        return row
    theta=counts/exposures
    hat,J=_risk_and_gradient(theta,N,i0)
    covdiag=theta*theta/counts
    var=np.einsum('ij,j,ij->i',J,covdiag,J)
    se=np.sqrt(np.maximum(0,var))
    if not (np.isfinite(theta).all() and np.isfinite(hat).all()
            and np.isfinite(se).all() and np.min(se)>1e-11):
        row['reason']='nonfinite or degenerate original standard errors'
        row['wall_seconds']=time.perf_counter()-t0
        return row

    row['valid']=True
    for j,name in enumerate(PARAM_NAMES):
        row['hat_'+name]=float(theta[j])
        row['estimated_var_'+name]=float(covdiag[j])
    row['parameter_cov_trace']=float(np.sum(covdiag))

    s=N-i0
    subset_ix,subset_c=_subset_indices(N,i0)
    H=hat[subset_ix]; T=risk0[subset_ix]; S=se[subset_ix]
    tail=np.arange(s)>=math.ceil(s*.5)
    row['maxT_empirical_full']=float(np.max(np.abs((hat-risk0)/se)))
    row['maxT_empirical_subset']=float(np.max(np.abs((H-T)/S)))
    for k,c in enumerate(subset_c):
        row[f'risk_hat_c{c}']=float(H[k])
        row[f'se2_c{c}']=float(S[k]**2)
        row[f'standardized_error_c{c}']=float((H[k]-T[k])/S[k])

    z=norm.ppf(1-alpha/2)
    bon=norm.ppf(1-alpha/(2*s))
    qgauss=_gaussian_critical(J,np.diag(covdiag),se,gaussian_draws,alpha)
    for name,q in [('Pointwise Wald',z),('Bonferroni',bon),
                   ('Gaussian max-t',qgauss)]:
        _metrics_for_critical(row,name,q,hat,risk0,se,tail)
    # Recalibrate the subset methods from the subset itself, NEVER by
    # reusing the full-curve bootstrap/Gaussian cutoff.
    qs_gauss=_gaussian_critical(J[subset_ix,:],np.diag(covdiag),S,gaussian_draws,alpha)
    for name,q in [('Pointwise Wald',z),
                   ('Bonferroni',norm.ppf(1-alpha/(2*len(subset_ix)))),
                   ('Gaussian max-t',qs_gauss)]:
        _subset_band(row,name,q,H,T,S)

    seeds=rng.integers(1,2**31-1,size=B,dtype=np.int64)
    fits,boot_counts=_boot_mles_counts(m,B,theta,N,i0,seeds)
    curves=_boot_risks_allow_boundary(fits,N,i0)
    valid_curve=np.isfinite(curves).all(axis=1)
    row['n_boot_valid_fixed_se']=int(valid_curve.sum())
    row['frac_boot_boundary']=float(np.mean((boot_counts<=0).any(axis=1)))
    row['frac_boot_invalid_fixed_se']=float(np.mean(~valid_curve))

    if valid_curve.sum()>=math.ceil(min_boot_frac*B):
        diff=curves[valid_curve]-hat[None,:]
        tstar=np.max(np.abs(diff/se[None,:]),axis=1)
        ts_subset=np.max(np.abs(diff[:,subset_ix]/S[None,:]),axis=1)
        qboot=_finite_q(tstar,alpha)
        qboot_s=_finite_q(ts_subset,alpha)
        row['tstar_fixed_median']=float(np.median(tstar))
        row['tstar_fixed_p99']=float(np.quantile(tstar,.99))
        row['tstar_fixed_max']=float(np.max(tstar))
        row['tstar_subset_p99']=float(np.quantile(ts_subset,.99))
    else:
        qboot=np.nan;qboot_s=np.nan
        row['bootstrap_problem']='too few valid risk curves for fixed-SE bootstrap'
    _metrics_for_critical(row,'Bootstrap max-t',qboot,hat,risk0,se,tail)
    _subset_band(row,'Bootstrap max-t',qboot_s,H,T,S)

    # Fully re-studentized pivot: keep bootstrap SE validation separate.
    _,boot_se=_boot_risk_ses(fits,boot_counts,N,i0)
    restud_ok=valid_curve & np.isfinite(boot_se).all(axis=1)
    restud_ok &= np.min(np.where(np.isfinite(boot_se),boot_se,np.inf),axis=1)>1e-11
    row['n_boot_valid_restudentized']=int(np.sum(restud_ok))
    row['frac_boot_invalid_restudentized']=float(np.mean(~restud_ok))
    if restud_ok.sum()>=math.ceil(min_boot_frac*B):
        ratios=np.abs((curves[restud_ok]-hat[None,:])/boot_se[restud_ok])
        tr=np.max(ratios,axis=1)
        trs=np.max(ratios[:,subset_ix],axis=1)
        qrest=_finite_q(tr,alpha)
        qrest_s=_finite_q(trs,alpha)
        row['tstar_restudentized_p99']=float(np.quantile(tr,.99))
        row['tstar_restudentized_max']=float(np.max(tr))
        row['median_bootstrap_se_ratio']=float(np.median(boot_se[restud_ok]/se[None,:]))
        # Diagnostic of destabilization from tiny bootstrap SEs.
        row['p01_boot_se_ratio']=float(np.quantile(boot_se[restud_ok]/se[None,:],.01))
    else:
        qrest=np.nan;qrest_s=np.nan
    _metrics_for_critical(row,'Bootstrap max-t (restudentized)',qrest,hat,risk0,se,tail)
    _subset_band(row,'Bootstrap max-t (restudentized)',qrest_s,H,T,S)

    # Existing log-rate linearization: diagnostic, not the paper's nonlinear bootstrap.
    goodlog=valid_curve & (fits>0).all(axis=1)
    row['n_boot_valid_log_rate']=int(goodlog.sum())
    if goodlog.sum()>=math.ceil(min_boot_frac*B):
        eta=np.log(fits[goodlog]/theta[None,:]);grad_eta=J*theta[None,:]
        difflog=eta@grad_eta.T
        tlog=np.max(np.abs(difflog/se[None,:]),axis=1)
        tlogs=np.max(np.abs(difflog[:,subset_ix]/S[None,:]),axis=1)
        qlog=_finite_q(tlog,alpha)
        qlog_s=_finite_q(tlogs,alpha)
        row['tstar_log_p99']=float(np.quantile(tlog,.99))
    else:
        qlog=np.nan;qlog_s=np.nan
    _metrics_for_critical(row,'Log-rate linearized bootstrap max-t',qlog,hat,risk0,se,tail)
    _subset_band(row,'Log-rate linearized bootstrap max-t',qlog_s,H,T,S)

    # The logit risk-scale method is a supplementary sensitivity check ONLY
    # on the prespecified subset (chosen before the MC experiment).
    if valid_curve.sum()>=math.ceil(min_boot_frac*B):
        se_logit=S/(np.clip(H,LOGIT_EPS,1-LOGIT_EPS)*(1-np.clip(H,LOGIT_EPS,1-LOGIT_EPS)))
        tz=np.max(np.abs((_logit(curves[valid_curve][:,subset_ix])-_logit(H)[None,:])
                         /se_logit[None,:]),axis=1)
        q_z=_finite_q(tz,alpha)
        row['logit_boot_extreme_frac']=float(np.mean(np.abs(tz)>20))
    else:
        q_z=np.nan
    _logit_subset_band(row,q_z,H,T,S)

    if valid_curve.any():
        pos=valid_curve & (fits>0).all(axis=1)
        if pos.any():
            zabs=np.max(np.abs(np.log(fits[pos]/theta[None,:])),axis=1)
            row['frac_boot_3x_parameter_outlier']=float(np.mean(zabs>np.log(3)))
    row['wall_seconds']=time.perf_counter()-t0
    return row


def _export(root,rows,N=50,i0=2,theta0=BASE_THETA,alpha=.05,show=True):
    if not rows:return
    df=pd.DataFrame(rows)
    df.to_csv(root/'simulation1_replications.csv',index=False)
    subset_ix,subset_c=_subset_indices(N,i0)
    tables=[];subtables=[]
    for m,grp in df.groupby('m',sort=True):
        valid=grp[grp.valid==True]
        for method in METHODS:
            eligible=valid[valid['valid_'+method]==True] if ('valid_'+method) in valid.columns else valid.iloc[:0]
            rec={'m':int(m),'method':method,'total_MC':len(grp),
                 'valid_original_MC':len(valid),'failed_original_MC':len(grp)-len(valid),
                 'valid_method_MC':len(eligible),'failed_method_MC':len(grp)-len(eligible)}
            if len(eligible):
                for suffix,key in [('full','Full_coverage'),('tail','Tail_coverage'),
                                   ('width','Average_width'),('clipped_width','Average_clipped_width'),
                                   ('point','Average_pointwise_coverage'),('missed','Mean_missed_thresholds')]:
                    a=eligible[method+'_'+suffix].astype(float).to_numpy()
                    rec[key]=float(a.mean())
                    if suffix=='full':
                        rec['Unconditional_full_coverage_failed_as_miss']=float(a.sum()/len(grp))
                    if suffix in ('full','tail'):
                        rec[key+'_MCSE']=float(np.std(a,ddof=1)/np.sqrt(len(a))) if len(a)>1 else np.nan
            tables.append(rec)
        for method in METHODS+[SUBSET_LOGIT_METHOD]:
            eligible=valid[valid['valid_subset_'+method]==True] if ('valid_subset_'+method) in valid else valid.iloc[:0]
            rec={'m':int(m),'method':method,'thresholds':','.join(map(str,subset_c)),
                 'total_MC':len(grp),'valid_method_MC':len(eligible),
                 'failed_method_MC':len(grp)-len(eligible)}
            if len(eligible):
                for suffix,key in [('full','Full_coverage'),('width','Average_width'),
                                   ('clipped_width','Average_clipped_width'),
                                   ('point','Average_pointwise_coverage'),('missed','Mean_missed_thresholds')]:
                    a=eligible[method+'_subset_'+suffix].astype(float).to_numpy()
                    rec[key]=float(a.mean())
                    if suffix=='full':
                        rec['Unconditional_full_coverage_failed_as_miss']=float(a.sum()/len(grp))
                        rec['Full_coverage_MCSE']=float(np.std(a,ddof=1)/np.sqrt(len(a))) if len(a)>1 else np.nan
            subtables.append(rec)
    summary=pd.DataFrame(tables)
    subset_summary=pd.DataFrame(subtables)
    summary.to_csv(root/'simulation1_summary.csv',index=False)
    subset_summary.to_csv(root/'simulation1_subset_summary.csv',index=False)

    diag_cols=['n_boot_valid_fixed_se','n_boot_valid_restudentized','n_boot_valid_log_rate',
               'frac_boot_boundary','frac_boot_invalid_fixed_se',
               'frac_boot_invalid_restudentized',
               'tstar_fixed_median','tstar_fixed_p99','tstar_fixed_max',
               'tstar_restudentized_p99','tstar_restudentized_max',
               'tstar_subset_p99','median_bootstrap_se_ratio','p01_boot_se_ratio',
               'frac_boot_3x_parameter_outlier','logit_boot_extreme_frac']
    diag_cols=[c for c in diag_cols if c in df]
    if diag_cols:
        df[df.valid==True].groupby('m')[diag_cols].agg(['mean','median','max']).to_csv(root/'bootstrap_diagnostics.csv')
    df[df.valid!=True][['m','rep','reason','minimum_original_count']].to_csv(root/'failed_original_MC.csv',index=False)

    covariance=[];rvar=[];maxcal=[]
    truthrisk=_risk_curve(theta0,N,i0)
    for m,grp in df.groupby('m'):
        valid=grp[grp.valid==True]
        if len(valid)<3:continue
        ph=valid[['hat_'+p for p in PARAM_NAMES]].to_numpy(dtype=float)
        emp=np.cov(ph,rowvar=False,ddof=1)
        model=np.diag(valid[['estimated_var_'+p for p in PARAM_NAMES]].mean().to_numpy(dtype=float))
        for j,p in enumerate(PARAM_NAMES):
            covariance.append({'m':int(m),'parameter':p,'n_valid_MC':len(valid),
                               'true_value':float(theta0[j]),
                               'mean_estimate':float(np.mean(ph[:,j])),
                               'bias':float(np.mean(ph[:,j])-theta0[j]),
                               'empirical_variance':float(emp[j,j]),
                               'mean_MLE_variance':float(model[j,j]),
                               'variance_ratio_empirical_to_MLE':float(emp[j,j]/model[j,j]) if model[j,j]>0 else np.nan})
        pd.DataFrame(emp,index=PARAM_NAMES,columns=PARAM_NAMES).to_csv(root/f'covariance_empirical_m{m}.csv')
        pd.DataFrame(model,index=PARAM_NAMES,columns=PARAM_NAMES).to_csv(root/f'covariance_mean_MLE_m{m}.csv')
        for c in subset_c:
            v=valid[f'risk_hat_c{c}'].to_numpy(dtype=float)
            estimated_se2=valid[f'se2_c{c}'].to_numpy(dtype=float)
            rvar.append({'m':int(m),'threshold':int(c),'n_valid_MC':len(valid),
                         'true_risk':float(truthrisk[c-1]),'mean_estimated_risk':float(v.mean()),
                         'risk_bias':float(v.mean()-truthrisk[c-1]),
                         'empirical_risk_variance':float(np.var(v,ddof=1)),
                         'mean_delta_variance':float(np.mean(estimated_se2)),
                         'variance_ratio_empirical_to_delta':float(np.var(v,ddof=1)/np.mean(estimated_se2))})
        for domain,key in [('full','maxT_empirical_full'),('subset','maxT_empirical_subset')]:
            errors=valid[key].to_numpy(dtype=float)
            observed95=float(np.quantile(errors,1-alpha))
            maxcal.append({'m':int(m),'domain':domain,'valid_MC':len(valid),
                           'empirical_maxT_q95':observed95,
                           'mean_Gaussian_q':float(valid['q_Gaussian max-t'].mean()) if domain=='full' else float(valid['q_subset_Gaussian max-t'].mean()),
                           'mean_bootstrap_fixed_q':float(valid['q_Bootstrap max-t'].mean()) if domain=='full' else float(valid['q_subset_Bootstrap max-t'].mean()),
                           'mean_bootstrap_restudentized_q':float(valid['q_Bootstrap max-t (restudentized)'].mean()) if domain=='full' else float(valid['q_subset_Bootstrap max-t (restudentized)'].mean()),
                           'empirical_exceedance_Gaussian_q':float(np.mean(errors > valid['q_Gaussian max-t'].to_numpy())) if domain=='full' else float(np.mean(errors > valid['q_subset_Gaussian max-t'].to_numpy()))})
    pd.DataFrame(covariance).to_csv(root/'covariance_diagnostics.csv',index=False)
    pd.DataFrame(rvar).to_csv(root/'threshold_variance_diagnostics.csv',index=False)
    pd.DataFrame(maxcal).to_csv(root/'maxT_calibration_diagnostics.csv',index=False)
    if show:
        cols=['m','method','valid_method_MC','failed_method_MC','Full_coverage','Full_coverage_MCSE','Average_width','Average_clipped_width']
        print('\nMAIN FULL-CURVE 95% SIMULTANEOUS COVERAGE:')
        print(summary.reindex(columns=cols).to_string(index=False))
        print(f'\nPRESPECIFIED SUBSET thresholds={list(map(int,subset_c))}:')
        print(subset_summary.reindex(columns=cols).to_string(index=False))
        print('\nMax-T calibration:')
        print(pd.DataFrame(maxcal).round(3).to_string(index=False))
        print('Output:',root)


def run(root,mode='new',M=DEFAULT_M,MC=2000,B=999,N=50,i0=2,
        alpha=.05,seed=20261010,gaussian=4000,threads=2,
        save_every=20,show_every=25,min_boot_frac=.90):
    if MC<1 or B<19 or N<=i0 or i0<=0: raise ValueError('Invalid MC, B, or population settings')
    numba.set_num_threads(min(max(1,int(threads)),numba.config.NUMBA_NUM_THREADS))
    cfg={'version':'2026-10-10-v4-fixed-subset-logit-gauss-diagnostics',
         'm':list(M),'MC':int(MC),'B':int(B),'N':int(N),'i0':int(i0),'alpha':float(alpha),
         'seed':int(seed),'gaussian':int(gaussian),'min_boot_frac':float(min_boot_frac),
         'threshold_fractions':list(THRESHOLD_FRACTIONS),'logit_epsilon':LOGIT_EPS}
    root=pathlib.Path(root)
    state,rng,ck=_start_or_resume(root,cfg,mode,seed)
    processed={(int(x['m']),int(x['rep'])) for x in state['rows']}
    # Warm up kernels: Numba compilation occurs only once per Colab runtime.
    warm=time.perf_counter()
    _one_dataset(1,BASE_THETA,8,2,1205)
    _boot_mles_counts(1,2,BASE_THETA,8,2,np.array([105,106],dtype=np.int64))
    _risk_and_gradient(BASE_THETA,8,2)
    _boot_risks_allow_boundary(np.array([BASE_THETA]),8,2)
    _boot_risk_ses(np.array([BASE_THETA]),np.ones((1,6)),8,2)
    print(f'[READY] CPU compilation {time.perf_counter()-warm:.1f}s; threads={numba.get_num_threads()}')
    rngG=np.random.default_rng(seed+98989)
    gaussian_draws=rngG.standard_normal((gaussian,6))
    risk0=_risk_curve(BASE_THETA,N,i0)
    start=time.perf_counter()
    target=len(M)*MC
    try:
        for m in M:
            for rep in range(MC):
                if (m,rep) in processed: continue
                result=_one_replication(BASE_THETA,risk0,int(m),N,i0,B,alpha,rng,
                                        gaussian_draws,min_boot_frac)
                result['rep']=int(rep)
                state['rows'].append(result)
                n=len(state['rows'])
                if n%save_every==0:
                    _save(state,rng,ck,root)
                if n%show_every==0 or n==1:
                    print(f'[MC] {n}/{target} | m={m} | rep={rep+1}/{MC} '
                          f'| {result["wall_seconds"]:.3f}s '
                          f'| {"OK" if result["valid"] else result["reason"]}',flush=True)
            _save(state,rng,ck,root)
            _export(root,state['rows'],N,i0,BASE_THETA,alpha)
        _save(state,rng,ck,root)
        _export(root,state['rows'],N,i0,BASE_THETA,alpha)
        print(f'\nFINISHED: {len(state["rows"])}/{target} MC in {time.perf_counter()-start:.1f}s this session.')
    except KeyboardInterrupt:
        _save(state,rng,ck,root)
        _export(root,state['rows'],N,i0,BASE_THETA,alpha)
        print('Interrupted; checkpoint saved. Choose RESUME next time.')
        raise
    except Exception:
        _save(state,rng,ck,root)
        _export(root,state['rows'],N,i0,BASE_THETA,alpha)
        raise


def _home():
    if 'google.colab' in sys.modules:
        from google.colab import drive
        drive.mount('/content/drive',force_remount=False)
        return pathlib.Path('/content/drive/MyDrive')
    return pathlib.Path.cwd()


def interactive():
    print('\n'+'='*62)
    print('SIMULATION 1 — FAST COMPLETE-DATA CPU v4')
    print('m=50,100,250,500 | B=999 | full curve + prespecified subset + logit')
    print('1 — START NEW (creates separate run folder)')
    print('2 — RESUME a previous run')
    print('Subset: 10%,20%,40%,60% of initially susceptible population')
    print('='*62)
    choice=input('Select 1 or 2: ').strip()
    if choice not in ('1','2'): raise ValueError('Choose 1 or 2')
    home=_home()
    runs=home/'Epidemic_Risk_Sim1_CPU_v4_runs'
    if choice=='1':
        print('Monte Carlo budget:')
        print('  1 — QUICK: 500 MC per m (2,000 total)')
        print('  2 — PRECISE: 2,000 MC per m (8,000 total)')
        preset=input('Select 1 or 2 [default 2]: ').strip() or '2'
        if preset not in ('1','2'): raise ValueError('Choose 1 or 2')
        mc=500 if preset=='1' else 2000
        root=runs/datetime.now().strftime('run_%Y%m%d_%H%M%S_%f')
        print('New run:',root)
        run(root,'new',MC=mc)
    else:
        options=[]
        for p in sorted(runs.glob('run_*')):
            ck=p/'checkpoint.pkl'; conf=p/'config.json'
            if ck.exists() and conf.exists():
                try:
                    cfg=json.loads(conf.read_text())
                    if cfg.get('version')=='2026-10-10-v4-fixed-subset-logit-gauss-diagnostics':
                        options.append((ck.stat().st_mtime,p,cfg))
                except (ValueError,OSError):pass
        options.sort(reverse=True,key=lambda a:a[0])
        if not options:
            print('No v4 checkpoint found. Start a new run with option 1.')
            return
        for k,(_,p,c) in enumerate(options,1):
            print(f'  {k}: {p.name} | {c["MC"]} MC per m')
        v=input('Checkpoint number [Enter=most recent]: ').strip()
        index=int(v)-1 if v else 0
        if index<0 or index>=len(options):raise ValueError('Invalid checkpoint number')
        _,root,cfg=options[index]
        print('Resuming:',root)
        run(root,'resume',M=cfg['m'],MC=cfg['MC'],B=cfg['B'],N=cfg['N'],i0=cfg['i0'],
            alpha=cfg['alpha'],seed=cfg['seed'],gaussian=cfg['gaussian'],
            min_boot_frac=cfg['min_boot_frac'])


def cli():
    parser=argparse.ArgumentParser(description='Simulation 1 CPU v4 — complete-data CTMC')
    parser.add_argument('--mode',choices=['new','resume'],default='new')
    parser.add_argument('--output',required=True)
    parser.add_argument('--m',nargs='+',type=int,default=DEFAULT_M)
    parser.add_argument('--MC',type=int,default=2000)
    parser.add_argument('--B',type=int,default=999)
    parser.add_argument('--N',type=int,default=50)
    parser.add_argument('--gaussian',type=int,default=4000)
    parser.add_argument('--threads',type=int,default=2)
    parser.add_argument('--seed',type=int,default=20261010)
    parser.add_argument('--save-every',type=int,default=20)
    parser.add_argument('--show-every',type=int,default=25)
    args=parser.parse_args()
    run(args.output,args.mode,M=args.m,MC=args.MC,B=args.B,N=args.N,
        gaussian=args.gaussian,threads=args.threads,seed=args.seed,
        save_every=args.save_every,show_every=args.show_every)


if __name__=='__main__':
    # The notebook invokes interactive() directly, rather than relying on argv.
    if any(a.startswith('--') for a in sys.argv[1:]) and 'ipykernel' not in sys.modules:
        cli()
    else:
        interactive()



SIMULATION 1 — FAST COMPLETE-DATA CPU v4
m=50,100,250,500 | B=999 | full curve + prespecified subset + logit
1 — START NEW (creates separate run folder)
2 — RESUME a previous run
Subset: 10%,20%,40%,60% of initially susceptible population
Select 1 or 2: 2
Mounted at /content/drive
  1: run_20261010_201657_145347 | 2000 MC per m
Checkpoint number [Enter=most recent]: 1
Resuming: /content/drive/MyDrive/Epidemic_Risk_Sim1_CPU_v4_runs/run_20261010_201657_145347
[READY] CPU compilation 8.9s; threads=2
[MC] 475/8000 | m=50 | rep=475/2000 | 2.558s | OK
[MC] 500/8000 | m=50 | rep=500/2000 | 2.071s | OK
[MC] 525/8000 | m=50 | rep=525/2000 | 2.071s | OK
[MC] 550/8000 | m=50 | rep=550/2000 | 2.413s | OK
[MC] 575/8000 | m=50 | rep=575/2000 | 1.733s | OK
[MC] 600/8000 | m=50 | rep=600/2000 | 2.439s | OK
[MC] 625/8000 | m=50 | rep=625/2000 | 2.558s | OK
[MC] 650/8000 | m=50 | rep=650/2000 | 2.079s | OK
[MC] 675/8000 | m=50 | rep=675/2000 | 1.688s | OK
[MC] 700/8000 | m=50 | rep=700/2000 | 2.783s | 

In [ ]:
"""FAST, PARTIALLY OBSERVED TWO-REGIME SIR SIMULATION 1 (CPU).

MODEL: Exact latent CTMC SIR paths, weekly binomial-reported incident cases.
INITIAL REGIME: observed at t=0, no regime or event path thereafter.
INFERENCE: NB *working* likelihood calibrated to EXACT CTMC expected weekly
incidence (computed on a pretabulated parameter grid). Beta1/beta2 fitted;
gamma1/gamma2/q12/q21, N, initial regime and reporting probability known.
This is NOT the exact partially observed hidden-state likelihood and NOT
six-parameter complete-data MLE; do not present as such.

COMPARISONS: pointwise Wald, Bonferroni, dependence-aware Gaussian max-t,
NB-working-model parametric bootstrap max-t, and stratified outbreak-level
nonparametric bootstrap max-t (recommended for genuine CTMC reported data).
Bootstrap uses refitting of the working likelihood, accelerated by a vectorized
Newton step and splines for exact CTMC mean/risk maps.

Colab: paste the ENTIRE script into ONE cell and run. Interactive start/resume.
Requires numpy, scipy, pandas, numba; CPU only.
"""
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
import json, math, time, pickle, pathlib, sys, hashlib, traceback
from datetime import datetime
import numpy as np
import pandas as pd
from scipy.interpolate import RectBivariateSpline
from scipy.optimize import minimize
from scipy.stats import norm
from numba import njit

CFG = dict(
    version='partial-ctmc-weekly-v3',
    N=30, i0=2, T=9, dt=1.0,
    theta_true=[1.45,0.75,1.,1.,0.35,0.25],
    m_values=[50,100,250,500], reports=[0.90,0.55],
    MC=500, B=999, alpha=0.05, gaussian_draws=2500,
    phi=10.0, seed=20261010, grid_points=36,
    restudentize=True,
    beta1_limits=[0.65,2.35], beta2_limits=[0.28,1.65],
    checkpoint_every=10, show_every=10,
    # 'ctmc' is the main genuine latent-path simulation.
    # 'nb' is a correctly specified working-likelihood diagnostic.
    experiment='ctmc',
    # Initial regime is observed at baseline; all subsequent regime changes hidden.
    observe_initial_regime=True,
    # Be explicit: no full 6-dimensional inference is claimed.
    fitted_parameters=['beta1','beta2'],
    threshold_true_risk_min=0.0005,
    threshold_true_risk_max=0.9995,
)

@njit(cache=True)
def _setup(N,i0):
    smax=N-i0
    K=(smax+1)*(2*N-smax+2) # upper bound; return trimmed
    n=0
    for s in range(smax+1):
        for i in range(N-s+1): n+=2
    sr=np.empty(n,np.float64); br=np.empty(n,np.float64)
    phase=np.empty(n,np.int64)
    ti=np.full(n,-1,np.int64); tr=np.full(n,-1,np.int64); tq=np.empty(n,np.int64)
    # explicit index[s,i,p]
    look=np.full((smax+1,N+1,2),-1,np.int64)
    z=0
    for s in range(smax+1):
        for i in range(N-s+1):
            for p in range(2):
                look[s,i,p]=z
                sr[z]=s
                br[z]=s*i/N
                phase[z]=p
                z+=1
    for s in range(smax+1):
        for i in range(N-s+1):
            for p in range(2):
                k=look[s,i,p]
                if s>0 and i>0: ti[k]=look[s-1,i+1,p]
                if i>0: tr[k]=look[s,i-1,p]
                tq[k]=look[s,i,1-p]
    a=np.zeros(n,np.float64); b=np.zeros(n,np.float64)
    a[look[smax,i0,0]]=1.
    b[look[smax,i0,1]]=1.
    return sr,br,phase,ti,tr,tq,a,b

@njit(cache=True)
def _one_step(v,rates,ti,tr,tq,nu,dt):
    # Uniformization for exp(Q^T dt) @ v, using a positive Poisson mixture.
    z=nu*dt
    w=math.exp(-z)
    out=w*v
    cur=v.copy()
    cumulative=w
    H=int(z+18.*math.sqrt(z+1.)+100.)
    for it in range(1,H+1):
        nxt=np.zeros(len(v),np.float64)
        for k in range(len(v)):
            mass=cur[k]
            if mass==0.:continue
            ri=rates[k,0]/nu; rr=rates[k,1]/nu; rq=rates[k,2]/nu
            nxt[k]+=mass*(1-ri-rr-rq)
            if ti[k]>=0:nxt[ti[k]]+=mass*ri
            if tr[k]>=0:nxt[tr[k]]+=mass*rr
            nxt[tq[k]]+=mass*rq
        cur=nxt
        w*=z/it
        out+=w*cur
        cumulative+=w
        if cumulative>1-2e-13 and it>z: break
    return out/cumulative

@njit(cache=True)
def _build_rates(beta1,beta2,theta,setup,N):
    sus,base,p,ti,tr,tq,a,b,infect=setup
    rates=np.empty((len(sus),3),np.float64)
    nu=0.
    for k in range(len(sus)):
        regime=p[k]
        rates[k,0]=(beta1 if regime==0 else beta2)*base[k]
        rates[k,1]=theta[2+regime]*infect[k]
        rates[k,2]=theta[4+regime]
        rate=rates[k,0]+rates[k,1]+rates[k,2]
        if rate>nu:nu=rate
    return rates,nu*1.00000001

@njit(cache=True)
def _weekly_means(beta1,beta2,theta,T,dt,setup,N):
    rates,nu=_build_rates(beta1,beta2,theta,setup,N)
    sus,base,p,ti,tr,tq,a,b,infect=setup
    output=np.empty((2,T),np.float64)
    for regime in range(2):
        v=a.copy() if regime==0 else b.copy()
        old=np.dot(sus,v)
        for t in range(T):
            v=_one_step(v,rates,ti,tr,tq,nu,dt)
            new=np.dot(sus,v)
            output[regime,t]=max(old-new,1e-13)
            old=new
    return output

@njit(cache=True)
def _risk_curve(theta,N,i0):
    smax=N-i0
    x=np.zeros((smax+1,N+1,smax+1,2),np.float64)
    for s in range(smax+1):
        for i in range(N-s+1):
            if i==0 or s==0:
                x[s,i,0,0]=1.
                x[s,i,0,1]=1.
                continue
            base=s*i/N
            inf1=theta[0]*base;inf2=theta[1]*base
            rec1=theta[2]*i;rec2=theta[3]*i
            a=inf1+rec1+theta[4];d=inf2+rec2+theta[5]
            det=a*d-theta[4]*theta[5]
            for n in range(s+1):
                h1=rec1*x[s,i-1,n,0]
                h2=rec2*x[s,i-1,n,1]
                if n>0:
                    h1+=inf1*x[s-1,i+1,n-1,0]
                    h2+=inf2*x[s-1,i+1,n-1,1]
                x[s,i,n,0]=(d*h1+theta[4]*h2)/det
                x[s,i,n,1]=(theta[5]*h1+a*h2)/det
    result=np.empty(smax,np.float64)
    acc=0.
    for n in range(smax,0,-1):
        acc+=x[smax,i0,n,0]
        result[n-1]=acc
    return result

@njit(cache=True)
def _sim_reported(m,theta,N,i0,T,dt,report,seed):
    np.random.seed(seed)
    y=np.zeros((m,T),np.int16)
    phases=np.empty(m,np.int8)
    b1,b2,g1,g2,q12,q21=theta
    pi0=q21/(q12+q21)
    for j in range(m):
        s=N-i0;i=i0
        regime=0 if np.random.random()<pi0 else 1
        phases[j]=regime
        t=0.
        while i>0 and t<T*dt:
            inf=(b1 if regime==0 else b2)*s*i/N
            rec=(g1 if regime==0 else g2)*i
            sw=q12 if regime==0 else q21
            rate=inf+rec+sw
            t+=-math.log(max(np.random.random(),1e-300))/rate
            if t>=T*dt:break
            u=np.random.random()*rate
            if u<inf:
                s-=1;i+=1
                if np.random.random()<report:
                    y[j,int(t/dt)]+=1
            elif u<inf+rec:
                i-=1
            else:regime=1-regime
    return y,phases

def setup_model(N,i0):
    sus,base,p,ti,tr,tq,a,b = _setup(N,i0)
    inf=np.zeros(len(sus),np.float64)
    # Recover number of infectives by tracking 2 states in original enumeration
    k=0
    for s in range(N-i0+1):
        for i in range(N-s+1):
            inf[k]=i;inf[k+1]=i;k+=2
    return (sus,base,p,ti,tr,tq,a,b,inf)

class FastSurrogate:
    def __init__(self,cfg,root):
        self.cfg=cfg;self.root=root
        self.N=cfg['N'];self.i0=cfg['i0'];self.T=cfg['T']
        self.theta=np.array(cfg['theta_true'],float)
        self.x=np.linspace(*cfg['beta1_limits'],cfg['grid_points'])
        self.z=np.linspace(*cfg['beta2_limits'],cfg['grid_points'])
        self.setup=setup_model(self.N,self.i0)
        cache=root/'ctmc_mean_risk_grid.npz'
        if cache.exists():
            with np.load(cache) as f:
                if f['x'].shape!=self.x.shape or not np.allclose(f['x'],self.x) or not np.allclose(f['z'],self.z):
                    raise RuntimeError('Incompatible grid cache')
                means=f['means']; risks=f['risks']
        else:
            start=time.perf_counter()
            means=np.empty((len(self.x),len(self.z),2,self.T))
            risks=np.empty((len(self.x),len(self.z),self.N-self.i0))
            for a,beta1 in enumerate(self.x):
                for b,beta2 in enumerate(self.z):
                    means[a,b]=_weekly_means(beta1,beta2,self.theta,self.T,cfg['dt'],self.setup,self.N)
                    th=self.theta.copy();th[0]=beta1;th[1]=beta2
                    risks[a,b]=_risk_curve(th,self.N,self.i0)
                if (a+1)%8==0:print(f'[GRID] {a+1}/{len(self.x)}; {time.perf_counter()-start:.1f}s',flush=True)
            np.savez_compressed(cache,means=means,risks=risks,x=self.x,z=self.z)
            print(f'[GRID] generated in {time.perf_counter()-start:.1f}s',flush=True)
        self.mean_spl=[]
        for p in range(2):
            for t in range(self.T):
                self.mean_spl.append(RectBivariateSpline(self.x,self.z,np.log(np.maximum(means[:,:,p,t],1e-13)),kx=3,ky=3,s=0))
        self.risk_spl=[RectBivariateSpline(self.x,self.z,risks[:,:,k],kx=3,ky=3,s=0) for k in range(risks.shape[-1])]
        self.cache_path=cache

    def mean(self,par,report,derivatives=False):
        p=np.asarray(par,dtype=float)
        if p.ndim==1:p=p[None,:]
        b1,b2=p[:,0],p[:,1]
        logs=np.stack([s.ev(b1,b2) for s in self.mean_spl],axis=-1).reshape(len(p),2,self.T)
        mu=report*np.exp(logs)
        if not derivatives:return mu
        g1=np.stack([s.ev(b1,b2,dx=1,dy=0) for s in self.mean_spl],axis=-1).reshape(len(p),2,self.T)
        g2=np.stack([s.ev(b1,b2,dx=0,dy=1) for s in self.mean_spl],axis=-1).reshape(len(p),2,self.T)
        g=np.stack([g1,g2],axis=-1)
        return mu,g

    def risk(self,par,derivatives=False):
        p=np.asarray(par,float)
        if p.ndim==1:p=p[None,:]
        x,y=p[:,0],p[:,1]
        risk=np.column_stack([s.ev(x,y) for s in self.risk_spl])
        if not derivatives:return risk
        g=np.stack([np.column_stack([s.ev(x,y,dx=1,dy=0) for s in self.risk_spl]),
                    np.column_stack([s.ev(x,y,dx=0,dy=1) for s in self.risk_spl])],axis=-1)
        return risk,g

    def validate(self,seed=100):
        rng=np.random.default_rng(seed)
        max_mu=0.;max_r=0.
        for _ in range(12):
            z=np.array([rng.uniform(self.x[2],self.x[-3]),rng.uniform(self.z[2],self.z[-3])])
            exact_mu=_weekly_means(*z,self.theta,self.T,self.cfg['dt'],self.setup,self.N)
            approx=self.mean(z,1.)[0]
            max_mu=max(max_mu,float(np.max(np.abs(exact_mu-approx))))
            th=self.theta.copy();th[:2]=z
            exact_r=_risk_curve(th,self.N,self.i0)
            max_r=max(max_r,float(np.max(np.abs(exact_r-self.risk(z)[0]))))
        print(f'[GRID VALIDATION] max absolute weekly mean error={max_mu:.3g}; risk error={max_r:.3g}')
        return max_mu,max_r

def negative_loglik_and_grad(par,totals,n,report,model,phi):
    mu,g=model.mean(par,report,True)
    mu,g=mu[0],g[0]
    f=phi
    # Constant-free NB loglik, aggregated over m independently observed curves
    loglike=np.sum(totals*np.log(mu/(phi+mu)) - n[:,None]*phi*np.log(phi+mu))
    A=totals-(n[:,None]*phi+totals)*mu/(phi+mu)
    grad=np.einsum('pt,ptd->d',A,g)
    return -float(loglike),-grad

def fit_single(totals,n,report,model,phi,start):
    bounds=[(model.x[1],model.x[-2]),(model.z[1],model.z[-2])]
    fit=minimize(negative_loglik_and_grad,np.asarray(start,float),args=(totals,n,report,model,phi),method='L-BFGS-B',jac=True,bounds=bounds,options={'maxiter':65,'ftol':1e-10,'gtol':1e-5})
    return fit

def fit_batch(totals,n,report,model,phi,start):
    """Vectorized damped Fisher-scoring fit, B independent NB working-likelihood refits."""
    totals=np.asarray(totals,float)
    if totals.ndim==2:totals=totals[None]
    B=len(totals)
    n=np.asarray(n,float)
    if n.ndim==1:n=np.broadcast_to(n,(B,2))
    x=np.broadcast_to(np.asarray(start,float),(B,2)).copy()
    low=np.array([model.x[1],model.z[1]])
    high=np.array([model.x[-2],model.z[-2]])
    x=np.minimum(np.maximum(x,low),high)
    damping=0.95
    for k in range(27):
        mu,g=model.mean(x,report,True)
        A=totals-(n[:,:,None]*phi+totals)*mu/(phi+mu)
        score=np.einsum('bpt,bptd->bd',A,g)
        weight=n[:,:,None]*mu*phi/(phi+mu)
        H=np.einsum('bpt,bptd,bpte->bde',weight,g,g)
        ridge=1e-8*(H[:,0,0]+H[:,1,1]+1.)
        H[:,0,0]+=ridge;H[:,1,1]+=ridge
        det=np.maximum(H[:,0,0]*H[:,1,1]-H[:,0,1]*H[:,1,0],1e-16)
        dx=np.column_stack([(H[:,1,1]*score[:,0]-H[:,0,1]*score[:,1])/det,
                            (H[:,0,0]*score[:,1]-H[:,1,0]*score[:,0])/det])
        dx=np.clip(dx,-.16,.16)*damping
        xnew=np.minimum(np.maximum(x+dx,low),high)
        x=xnew
        if np.max(np.linalg.norm(dx,axis=1))<1e-6:break
    mu,g=model.mean(x,report,True)
    A=totals-(n[:,:,None]*phi+totals)*mu/(phi+mu)
    score=np.einsum('bpt,bptd->bd',A,g)
    weight=n[:,:,None]*mu*phi/(phi+mu)
    H=np.einsum('bpt,bptd,bpte->bde',weight,g,g)
    det=H[:,0,0]*H[:,1,1]-H[:,0,1]**2
    scale=np.maximum(1,np.einsum('bpt->b',weight))
    normalized_score=score/scale[:,None]
    # Constrained MLE KKT: nonzero scores are legitimate at bounds.
    # Lower bound requires nonpositive score; upper bound nonnegative.
    free=(x>low+1e-6)&(x<high-1e-6)
    lower=x<=low+1e-6
    upper=x>=high-1e-6
    stationary=(free&(np.abs(normalized_score)<1e-3))|(lower&(normalized_score<=1e-3))|(upper&(normalized_score>=-1e-3))
    err=np.max(np.where(free,np.abs(normalized_score),0.),axis=1)
    ok=(np.isfinite(x).all(axis=1)&(det>1e-9)&stationary.all(axis=1))
    return x,ok,err

def scores_and_cov(y,p0,par,report,model,phi):
    mu,g=model.mean(par,report,True);mu=mu[0];g=g[0]
    n=np.bincount(p0,minlength=2)
    # Individual outbreak-level scores (robust to within-outbreak correlation)
    residual=(y-mu[p0])*phi/(phi+mu[p0])
    scores=np.einsum('jt,jtd->jd',residual,g[p0])
    scores=scores-scores.mean(axis=0,keepdims=True)
    H=np.einsum('pt,ptd,pte->de',n[:,None]*mu*phi/(phi+mu),g,g)
    eigen=np.linalg.eigvalsh(H)
    if eigen[0]<1e-8 or eigen[-1]/eigen[0]>1e9:return None
    Hinv=np.linalg.inv(H)
    sandwich=Hinv@(scores.T@scores)@Hinv
    modelcov=Hinv
    if np.linalg.eigvalsh(sandwich)[0]<1e-12:return None
    return sandwich,modelcov

def bootstrap_totals_cluster(y,p0,B,rng):
    out=np.zeros((B,2,y.shape[1]),float)
    all_weights=np.zeros((B,len(y)),np.int32)
    n=np.bincount(p0,minlength=2)
    for phase in range(2):
        ys=y[p0==phase].astype(float)
        if len(ys):
            weights=rng.multinomial(int(n[phase]),np.full(int(n[phase]),1./n[phase]),size=B)
            out[:,phase,:]=weights@ys
            all_weights[:,p0==phase]=weights
    return out,np.broadcast_to(n,(B,2)).copy(),all_weights

def bootstrap_totals_nb(m,phase_count,B,par,report,model,phi,rng):
    mu=model.mean(par,report)[0]
    totals=np.zeros((B,2,len(mu[0])),float)
    for p in range(2):
        if phase_count[p]>0:
            r=phase_count[p]*phi
            prob=phi/(phi+mu[p])
            totals[:,p,:]=rng.negative_binomial(r,prob,size=(B,len(mu[p])))
    return totals,np.broadcast_to(phase_count,(B,2)).copy()

@njit(cache=True)
def _bootstrap_sandwich(y,p0,weights,mu,g,phi):
    """Outbreak-cluster sandwich covariance for each stratified resample.

    Every bootstrap covariance is computed at its own refitted parameter,
    weighting whole original outbreaks by bootstrap multiplicities.
    """
    B=weights.shape[0];m=y.shape[0];T=y.shape[1]
    out=np.empty((B,2,2),np.float64)
    for b in range(B):
        h00=0.;h01=0.;h11=0.
        for p in range(2):
            n=0
            for j in range(m):
                if p0[j]==p:n+=weights[b,j]
            for t in range(T):
                factor=n*mu[b,p,t]*phi/(phi+mu[b,p,t])
                g0=g[b,p,t,0];g1=g[b,p,t,1]
                h00+=factor*g0*g0
                h01+=factor*g0*g1
                h11+=factor*g1*g1
        det=h00*h11-h01*h01
        if det<=1e-10:
            for i in range(2):
                for j in range(2):out[b,i,j]=np.nan
            continue
        v00=h11/det;v01=-h01/det;v11=h00/det
        o00=0.;o01=0.;o11=0.
        for j in range(m):
            w=weights[b,j]
            if w==0:continue
            p=p0[j];z0=0.;z1=0.
            for t in range(T):
                f=(y[j,t]-mu[b,p,t])*phi/(phi+mu[b,p,t])
                z0+=f*g[b,p,t,0]
                z1+=f*g[b,p,t,1]
            o00+=w*z0*z0;o01+=w*z0*z1;o11+=w*z1*z1
        a=v00*o00+v01*o01
        c=v00*o01+v01*o11
        b0=v01*o00+v11*o01
        d=v01*o01+v11*o11
        out[b,0,0]=a*v00+c*v01
        out[b,0,1]=a*v01+c*v11
        out[b,1,0]=b0*v00+d*v01
        out[b,1,1]=b0*v01+d*v11
    return out

def gaussian_critical(jac,se,cov,draws,rng,alpha):
    eig,vec=np.linalg.eigh(cov)
    root=(vec*np.sqrt(np.maximum(eig,0)))@vec.T
    sims=rng.standard_normal((draws,2))@root.T
    return float(np.quantile(np.max(np.abs(sims@jac.T/se),axis=1),1-alpha))

def assess_one(y,p0,report,model,cfg,truth,ids,rng):
    m=len(y);B=cfg['B'];phi=cfg['phi'];alpha=cfg['alpha']
    n=np.bincount(p0,minlength=2)
    totals=np.stack([y[p0==p].sum(axis=0) for p in range(2)])
    if np.min(n)<3:raise ValueError('Fewer than 3 trajectories in one initial-regime stratum')
    fit=fit_single(totals,n,report,model,phi,np.array(cfg['theta_true'][:2]))
    if not np.isfinite(fit.fun):raise ValueError('Nonfinite NB working likelihood')
    par=fit.x
    if np.any(par<=np.array([model.x[1],model.z[1]])) or np.any(par>=np.array([model.x[-2],model.z[-2]])):
        raise ValueError('Fit on parameter grid boundary')
    covs=scores_and_cov(y,p0,par,report,model,phi)
    if covs is None:raise ValueError('Singular working-information or sandwich covariance')
    sandwich,modelcov=covs
    curve,grad=model.risk(par,True);curve=curve[0,ids];grad=grad[0,ids]
    se=np.sqrt(np.maximum(np.einsum('ij,jk,ik->i',grad,sandwich,grad),0.))
    if np.min(se)<1e-7:raise ValueError('Near-zero risk standard error on fixed threshold set')
    error=np.abs(curve-truth[ids]);K=len(ids)
    crit={'Pointwise Wald':float(norm.ppf(1-alpha/2)),
          'Bonferroni':float(norm.ppf(1-alpha/(2*K))),
          'Gaussian max-t':gaussian_critical(grad,se,sandwich,cfg['gaussian_draws'],rng,alpha)}
    # Working parametric bootstrap (fast, but misspecified when true DGP is CTMC)
    nt,ns=bootstrap_totals_nb(m,n,B,par,report,model,phi,rng)
    bp,okp,errp=fit_batch(nt,ns,report,model,phi,par)
    # Stratified outbreak bootstrap, accounts for CTMC inter-week correlation.
    ct,cs,bootstrap_weights=bootstrap_totals_cluster(y,p0,B,rng)
    bc,okc,errc=fit_batch(ct,cs,report,model,phi,par)
    rows={'Valid bootstrap NB':int(okp.sum()),'Valid bootstrap cluster':int(okc.sum())}
    for label,bfit,ok in [('NB parametric bootstrap max-t',bp,okp),('Cluster bootstrap max-t',bc,okc)]:
        rows['fraction_valid_'+label]=float(ok.mean())
        if ok.sum()>=int(.90*B):
            curves=model.risk(bfit[ok])[...,ids]
            zz=np.max(np.abs((curves-curve)/se),axis=1)
            idx=min(len(zz)-1,math.ceil((len(zz)+1)*(1-alpha))-1)
            crit[label]=float(np.partition(zz,idx)[idx])
            rows['critical_'+label]=crit[label]
        else:
            # The entire MC is NOT lost: only this method becomes unavailable.
            # Its smaller valid bootstrap fraction remains in the raw CSV.
            rows['issue_'+label]=f'only {ok.sum()}/{B} valid bootstrap refits'
    # Optional second bootstrap variant: robust re-studentization at each
    # refitted cluster resample; diagnostic only (may be unstable for small m).
    if cfg.get('restudentize',False) and okc.sum()>=int(.90*B):
        valid_inds=np.where(okc)[0]
        c_mu,c_g=model.mean(bc[valid_inds],report,True)
        c_cov=_bootstrap_sandwich(y.astype(np.float64),p0.astype(np.int64),
                    bootstrap_weights[valid_inds],c_mu,c_g,phi)
        b_risk,b_grad=model.risk(bc[valid_inds],True)
        b_risk=b_risk[:,ids,:] if b_risk.ndim==3 else b_risk[:,ids]
        b_grad=b_grad[:,ids,:]
        vv=np.einsum('bki,bij,bkj->bk',b_grad,c_cov,b_grad)
        b_se=np.sqrt(np.maximum(vv,0))
        good=np.isfinite(b_se).all(axis=1)&(b_se>1e-7).all(axis=1)
        rows['Valid re-studentized cluster']=int(good.sum())
        if good.sum()>=int(0.85*B):
            zrest=np.max(np.abs((b_risk[good]-curve)/b_se[good]),axis=1)
            ix=min(len(zrest)-1,math.ceil((len(zrest)+1)*(1-alpha))-1)
            crit['Cluster bootstrap max-t (re-studentized)']=float(np.partition(zrest,ix)[ix])
        else:
            # No silent imputation or invalid critical value.
            rows['restudentization_issue']='insufficient finite studentization'
    rows.update(beta1_hat=float(par[0]),beta2_hat=float(par[1]),n_initial_phase1=int(n[0]),
                sandwich_condition=float(np.linalg.cond(sandwich)))
    for method,q in crit.items():
        hit=error<=q*se
        lower=np.maximum(0,curve-q*se);upper=np.minimum(1,curve+q*se)
        rows.update({method+'_full':int(hit.all()),method+'_point':float(hit.mean()),
                     method+'_missed':int((~hit).sum()),method+'_width':float(np.mean(2*q*se)),
                     method+'_clipped_width':float(np.mean(upper-lower)),
                     method+'_tail':int(hit[ids>=np.median(ids)].all())})
    return rows

def checkpoint(path,state,rng):
    state['rng_state']=rng.bit_generator.state
    tmp=path.with_suffix('.tmp')
    with open(tmp,'wb') as f:
        pickle.dump(state,f,protocol=pickle.HIGHEST_PROTOCOL)
        f.flush();os.fsync(f.fileno())
    os.replace(tmp,path)

def summary(rows,root,verbose=False):
    if not rows:return
    df=pd.DataFrame(rows)
    df.to_csv(root/'simulation1_partial_replicates.csv',index=False)
    methods=['Pointwise Wald','Bonferroni','Gaussian max-t','NB parametric bootstrap max-t','Cluster bootstrap max-t','Cluster bootstrap max-t (re-studentized)']
    agg=[]
    for (report,m),grp in df.groupby(['report','m'],sort=True):
        ok=grp[grp['valid']==True]
        for method in methods:
            if ok.empty or method+'_full' not in ok:continue
            eligible=ok[ok[method+'_full'].notna()]
            if eligible.empty:continue
            c=float(eligible[method+'_full'].mean())
            agg.append(dict(experiment='CTMC_latent_weekly_reported',reporting=report,m=m,method=method,
                valid_MC=len(eligible),failed_MC=len(grp)-len(eligible),full_coverage=c,
                full_coverage_MCSE=np.sqrt(c*(1-c)/len(eligible)),
                average_width=float(eligible[method+'_width'].mean()),
                average_clipped_width=float(eligible[method+'_clipped_width'].mean()),
                pointwise_coverage=float(eligible[method+'_point'].mean()),
                average_missed_thresholds=float(eligible[method+'_missed'].mean()),
                average_seconds=float(eligible['wall_seconds'].mean())))
    table=pd.DataFrame(agg)
    table.to_csv(root/'simulation1_partial_summary.csv',index=False)
    if verbose and not table.empty:
        print('\nPARTIALLY OBSERVED SUMMARY:\n'+table[['reporting','m','method','valid_MC','failed_MC','full_coverage','average_width']].to_string(index=False),flush=True)

def choose_root_and_mode(cfg):
    try:
        from google.colab import drive
        drive.mount('/content/drive',force_remount=False)
        base=pathlib.Path('/content/drive/MyDrive/Epidemic_Risk_Sim1_Partial_CPU')
    except ImportError:
        base=pathlib.Path.cwd()/'Epidemic_Risk_Sim1_Partial_CPU'
    base.mkdir(parents=True,exist_ok=True)
    print('\nSIMULATION 1 — PARTIALLY OBSERVED WEEKLY INCIDENCE / CPU')
    print('1 — Start NEW experiment in a NEW folder')
    print('2 — RESUME previous experiment')
    mode=input('Choose 1 or 2: ').strip()
    if mode not in ('1','2'):raise ValueError('Choose 1 or 2')
    if mode=='1':
        root=base/('run_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        root.mkdir()
    else:
        candidates=sorted([p for p in base.glob('run_*') if (p/'checkpoint.pkl').exists()])
        if not candidates:raise FileNotFoundError('No previous partial run checkpoints found in '+str(base))
        if len(candidates)>1:
            print('Available checkpoints:')
            for j,p in enumerate(candidates,1): print(f'  {j}. {p.name}')
            chosen=int(input(f'Select run (1-{len(candidates)}): '))
            root=candidates[chosen-1]
        else: root=candidates[0]
    print(f'Run folder: {root} | mode: {mode}')
    return root,mode

def main():
    import argparse
    parser=argparse.ArgumentParser(add_help=False)
    parser.add_argument('--smoke',action='store_true'); parser.add_argument('--mc',type=int)
    parser.add_argument('--B',type=int);parser.add_argument('--grid',type=int)
    args,_=parser.parse_known_args() # Colab ipykernel arguments safely ignored
    cfg=CFG.copy()
    if args.smoke:cfg.update(MC=2,B=29,grid_points=16,m_values=[50],reports=[.9])
    if args.mc is not None:cfg['MC']=args.mc
    if args.B is not None:cfg['B']=args.B
    if args.grid is not None:cfg['grid_points']=args.grid
    if cfg['grid_points']<10:raise ValueError('grid_points must be >= 10')
    root,mode=choose_root_and_mode(cfg)
    fp=hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()
    path=root/'checkpoint.pkl'
    if mode=='1':
        state={'fingerprint':fp,'rows':[],'rng_state':np.random.default_rng(cfg['seed']).bit_generator.state}
        (root/'configuration.json').write_text(json.dumps(cfg,indent=2))
    else:
        with open(path,'rb') as f:state=pickle.load(f)
        if state['fingerprint']!=fp:
            raise ValueError('Checkpoint has different settings. Use original settings or start a new run.')
    rng=np.random.default_rng()
    rng.bit_generator.state=state['rng_state']
    theta=np.array(cfg['theta_true'],float)
    tic=time.perf_counter()
    model=FastSurrogate(cfg,root)
    maxmu,maxrisk=model.validate()
    if maxmu>0.008 or maxrisk>0.002:
        raise RuntimeError('Spline error too large; increase grid_points and start a fresh run')
    true_risk=_risk_curve(theta,cfg['N'],cfg['i0'])
    ids=np.where((true_risk>cfg['threshold_true_risk_min'])&(true_risk<cfg['threshold_true_risk_max']))[0]
    # threshold set is FIXED before all Monte Carlo replications, never data-dependent
    print('Fixed severity thresholds:',(ids+1).tolist())
    if len(ids)<5:raise RuntimeError('Insufficient fixed risk thresholds')
    # smoke compile
    _sim_reported(2,theta,cfg['N'],cfg['i0'],cfg['T'],cfg['dt'],.9,1234)
    todo=len(cfg['m_values'])*len(cfg['reports'])*cfg['MC']
    existing={(float(r['report']),int(r['m']),int(r['rep'])) for r in state['rows']}
    print(f'Expected {todo} MC replications; completed {len(existing)}; B={cfg["B"]}; CPU only.')
    print('NOTE: Gamma and switching rates are known; beta1, beta2 are fitted by NB working likelihood.')
    print('Main CTMC bootstrap benchmark: STRATIFIED CLUSTER bootstrap, not exact CTMC parametric bootstrap.')
    counter=len(existing)
    try:
        for report in cfg['reports']:
            for m in cfg['m_values']:
                print(f'[SCENARIO] report={report}, m={m}',flush=True)
                for rep in range(cfg['MC']):
                    key=(float(report),int(m),int(rep))
                    if key in existing:continue
                    start=time.perf_counter()
                    # RNG seed is checkpointed ahead of sim via generator state.
                    seed=int(rng.integers(1,2**31-1))
                    y,p0=_sim_reported(m,theta,cfg['N'],cfg['i0'],cfg['T'],cfg['dt'],report,seed)
                    row=dict(report=float(report),m=int(m),rep=int(rep),valid=False,reason='')
                    try:
                        details=assess_one(y,p0,report,model,cfg,true_risk,ids,rng)
                        row.update(details);row['valid']=True
                    except (ValueError,ArithmeticError,FloatingPointError,np.linalg.LinAlgError) as exc:
                        row['reason']=str(exc)[:230]
                    row['wall_seconds']=time.perf_counter()-start
                    state['rows'].append(row)
                    counter+=1
                    # Drive checkpoint is saved every 10 MC; saves are atomic.
                    if counter%cfg['checkpoint_every']==0 or counter==todo:
                        checkpoint(path,state,rng)
                        summary(state['rows'],root)
                    if counter%cfg['show_every']==0 or counter==todo:
                        recent=np.mean([r['wall_seconds'] for r in state['rows'][-min(25,len(state['rows'])):]])
                        eta=(todo-counter)*recent/3600
                        print(f'[MC] completed={counter}/{todo} | last={row["wall_seconds"]:.2f}s | '
                              f'valid={row["valid"]} | recent={recent:.2f}s | optimistic ETA={eta:.2f}h',flush=True)
        checkpoint(path,state,rng);summary(state['rows'],root,verbose=True)
        print(f'FINISHED. Elapsed this session {time.perf_counter()-tic:.1f}s; folder={root}')
    except KeyboardInterrupt:
        checkpoint(path,state,rng);summary(state['rows'],root)
        print('Interrupted; restart and choose 2 to RESUME.');raise

if __name__=='__main__':main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

SIMULATION 1 — EXACT CTMC + RADEMACHER MULTIPLIER MAX-T
New experiment directory: /content/drive/MyDrive/Epidemic_Risk_Simulation1_Multiplier
New-method checkpoint: YES
Old parametric-bootstrap checkpoint: YES

1 — START NEW MULTIPLIER EXPERIMENT
2 — RESUME NEW MULTIPLIER CHECKPOINT
3 — IMPORT ONLY ACTIVE DATASET AND FIT FROM OLD CHECKPOINT
    (only available if the old run was still on the first case)


KeyboardInterrupt: Interrupted by user